# Text-to-Image & Text-to-Speech with Open Source Models

**Platform:** Google Colab (Tesla T4 GPU)
**Notebook:** [Open in Google Colab](https://colab.research.google.com/drive/1WTtl4Kl3yb6Bd6vr3zEJC_2oOXDTI5gz#scrollTo=ddp81KvxEtC0)

## Overview
This session explores multimodal generation using open-source models — image synthesis via Stable Diffusion XL variants and text-to-speech via SpeechT5 — running on Colab's free Tesla T4 GPU.

## 1. SDXL Turbo — Fast Image Generation
**Model:** `stabilityai/sdxl-turbo`
Uses Adversarial Diffusion Distillation (ADD) to generate images in as few as 1–4 steps (vs. 30–50 for standard diffusion). Runs with `guidance_scale=0.0` since Turbo models are trained without classifier-free guidance.

## 2. SDXL Base 1.0 — High-Fidelity Generation
**Model:** `stabilityai/stable-diffusion-xl-base-1.0`
Full 30-step SDXL pipeline for higher quality single-pass output, loaded with `use_safetensors=True`.

## 3. SDXL Base + Refiner — Two-Stage Expert Pipeline
**Models:** SDXL Base 1.0 + `stabilityai/stable-diffusion-xl-refiner-1.0`

| Stage | Role |
|---|---|
| Base model | Handles early, high-noise steps — establishes composition |
| Refiner model | Handles final, low-noise steps — adds detail & realism |

Configured with `n_steps=40`, `high_noise_frac=0.8` (base runs 80%, refiner runs final 20%). Base output stays in latent space (`output_type="latent"`) and passes directly to the refiner via `denoising_start`, avoiding a wasted decode/re-encode round trip.

## 4. SpeechT5 — Text-to-Speech
**Model:** `microsoft/speecht5_tts`
Uses the HF `pipeline("text-to-speech", ...)` API. Requires a speaker embedding (x-vector) to condition voice identity — sourced from `Matthijs/cmu-arctic-xvectors` (speaker index 7306).

## Key Takeaways
- **Speed vs. quality:** Turbo (4 steps) vs. Base (30 steps) vs. Base+Refiner (40 steps) spans the real-time-to-production-quality spectrum.
- **Latent-space handoff** between base and refiner is the key efficiency trick in expert-ensemble diffusion pipelines.
- **Conditioning via embeddings** (speaker x-vectors, text encoders) is a pattern that recurs across modalities.
- A free-tier T4 GPU handles all of the above at `fp16` precision, one pipeline at a time.